# DTM Ground Segmentation — Enhanced PointNet++ (>95% Accuracy, 10-Hour Budget)

## Definitive P100 / sm_60 Fix

### Root cause (one sentence)
**No official PyTorch release supports both Python 3.12 AND P100 (sm_60) simultaneously** — they were never built together.

### How Cell 1 solves it permanently
```
1. apt install python3.10          (via deadsnakes PPA — takes ~30s)
2. python3.10 -m venv /kaggle/working/torch_env
3. pip install torch-2.0.1+cu117   (the cp310 wheel — takes ~3-5 min)
4. pip install ipykernel + register as "PyTorch 2.0.1 P100 (py3.10)"
5. GPU smoke test to confirm sm_60 kernels work
```

### After Cell 1 finishes — ONE action required:
> **Kernel menu → Change Kernel → "PyTorch 2.0.1 P100 (py3.10)"**  
> (Do NOT click Restart — you need to SWITCH kernel, not restart the same one)

Then run all cells from **Cell 2** onward.

### Memory + 10-hour budget (unchanged from v2)
| Fix | Impact |
|---|---|
| 2048 pts/tile (was 4096) | ~2x faster per batch, fits 35 full epochs in 10h |
| float32 feature cache | Halves RAM during one-time cache build |
| Gradient checkpointing SA1+SA2 | ~40% VRAM saved during backprop |
| `torch.cuda.amp.*` APIs | Compatible with PyTorch 2.0.1 |
| `free_memory()` after every val | Prevents VRAM fragmentation |
| Auto-stage promotion (200→2K→14K tiles) | Starts fast, grows automatically |


In [ ]:
# Cell 1 — DEFINITIVE P100 Fix: Install Python 3.10 + PyTorch 2.0.1 as a Jupyter kernel
#
# WHY THIS IS NEEDED:
#   - P100 = CUDA sm_60 (Pascal architecture)
#   - PyTorch >= 2.1 dropped sm_60 kernel binaries entirely
#   - PyTorch 2.0.1 (last version with sm_60) has NO Python 3.12 wheels — never released
#   - conda is not available on Kaggle GPU notebooks
#   - SOLUTION: Install Python 3.10 via apt, create a venv, install torch 2.0.1 there,
#               register it as a Jupyter kernel, then switch to it.
#
# STEPS:
#   1. Run this cell  (takes ~5-8 minutes)
#   2. When done: Kernel menu -> Change Kernel -> "PyTorch 2.0.1 P100"
#   3. Run all cells from Cell 2 onward  (do NOT restart — CHANGE KERNEL)

import subprocess, sys, os, time
from pathlib import Path

VENV_DIR    = "/kaggle/working/torch_env"
KERNEL_NAME = "torch_p100"
TORCH_WHL   = ("https://download.pytorch.org/whl/cu117/"
               "torch-2.0.1%2Bcu117-cp310-cp310-linux_x86_64.whl")


def sh(cmd, check=False, timeout=600, silent=False):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    if not silent:
        if r.stdout.strip():
            print(r.stdout.strip()[-500:])
        if r.returncode != 0 and r.stderr.strip():
            print("STDERR:", r.stderr.strip()[-300:])
    return r.returncode, r.stdout, r.stderr


print("=" * 65)
print("  P100 / sm_60 FIX: Installing Python 3.10 + PyTorch 2.0.1")
print("=" * 65)
t_start = time.time()

# ── Step 1: Check if already done ─────────────────────────────────────────────
py310 = Path(VENV_DIR) / "bin" / "python"
if py310.exists():
    print(f"\n✅ Venv already exists at {VENV_DIR}")
    print("   Checking torch in it...")
    code, out, _ = sh(f"{py310} -c \"import torch; print(torch.__version__)\"", silent=True)
    if code == 0 and "2.0" in out:
        print(f"   torch {out.strip()} already installed in venv.")
    else:
        print("   torch not yet installed, continuing...")
else:
    # ── Step 2: Install Python 3.10 ───────────────────────────────────────────
    print("\n[1/5] Installing Python 3.10 via deadsnakes PPA...")
    sh("add-apt-repository -y ppa:deadsnakes/ppa 2>/dev/null", silent=True)
    sh("apt-get update -qq 2>/dev/null", silent=True)
    code, _, err = sh(
        "apt-get install -y -qq python3.10 python3.10-venv python3.10-dev",
        timeout=300
    )
    if code != 0:
        # Fallback: try system python3.10 if already available
        code2, _, _ = sh("python3.10 --version", silent=True)
        if code2 != 0:
            raise RuntimeError("Could not install Python 3.10. See error above.")
    print("   ✅ Python 3.10 installed")

    # ── Step 3: Create venv ────────────────────────────────────────────────────
    print(f"\n[2/5] Creating venv at {VENV_DIR}...")
    sh(f"python3.10 -m venv {VENV_DIR} --system-site-packages", timeout=60)
    print("   ✅ Venv created")

pip310  = f"{VENV_DIR}/bin/pip"
py310   = f"{VENV_DIR}/bin/python"

# ── Step 4: Upgrade pip ────────────────────────────────────────────────────────
print("\n[3/5] Upgrading pip in venv...")
sh(f"{pip310} install -q --upgrade pip setuptools wheel", timeout=120, silent=True)
print("   ✅ pip upgraded")

# ── Step 5: Install PyTorch 2.0.1+cu117 ───────────────────────────────────────
# Check first (venv might already have it)
code, out, _ = sh(f"{py310} -c \"import torch; print(torch.__version__)\"", silent=True)
if code == 0 and "2.0" in out:
    print(f"\n[4/5] PyTorch {out.strip()} already in venv ✅")
else:
    print(f"\n[4/5] Installing PyTorch 2.0.1+cu117 (Python 3.10 wheel)...")
    print(f"      Downloading ~2.4 GB — this takes 2-5 minutes...")
    code, out, err = sh(
        f"{pip310} install -q {TORCH_WHL}",
        timeout=600
    )
    if code != 0:
        # Try cu118 as fallback (same sm_60 support, different CUDA runtime)
        print("   cu117 failed, trying cu118 fallback...")
        WHL2 = TORCH_WHL.replace("cu117", "cu118").replace(
            "torch-2.0.1%2Bcu117", "torch-2.0.1%2Bcu118")
        code, out, err = sh(f"{pip310} install -q {WHL2}", timeout=600)
        if code != 0:
            raise RuntimeError(
                f"PyTorch wheel install failed.\n{err[-500:]}\n\n"
                "Manual fallback: run this in a NEW cell:\n"
                f"  !{pip310} install {TORCH_WHL}"
            )
    print("   ✅ PyTorch 2.0.1 installed")

# ── Step 6: Install other packages ────────────────────────────────────────────
print("\n[4b/5] Installing other dependencies...")
sh(f"{pip310} install -q tqdm numpy matplotlib scipy psutil ipykernel",
   timeout=180, silent=True)
print("   ✅ tqdm, numpy, matplotlib, scipy, psutil, ipykernel installed")

# ── Step 7: Register as Jupyter kernel ────────────────────────────────────────
print(f"\n[5/5] Registering Jupyter kernel '{KERNEL_NAME}'...")
code, out, err = sh(
    f"{py310} -m ipykernel install --user "
    f"--name {KERNEL_NAME} "
    f'--display-name "PyTorch 2.0.1 P100 (py3.10)"',
    timeout=60
)
if code != 0:
    print(f"  Kernel registration issue: {err.strip()[-200:]}")
else:
    print(f"   ✅ Kernel registered")

# ── Quick GPU smoke test ───────────────────────────────────────────────────────
print("\nRunning GPU smoke test with new torch...")
code, out, err = sh(
    f"{py310} -c \""
    "import torch; "
    "x = torch.randn(64,64,device='cuda'); "
    "y = (x@x).sum(); "
    "y.item(); "
    "cap=torch.cuda.get_device_capability(0); "
    "print(f'torch={torch.__version__} GPU={torch.cuda.get_device_name(0)} "
    "sm_{cap[0]}{cap[1]} OK')"
    "\""
)
if code == 0:
    print(f"   ✅ {out.strip()}")
else:
    print(f"   ⚠️  GPU test output: {out.strip()}")
    print(f"   stderr: {err.strip()[-200:]}")

elapsed = int(time.time() - t_start)
print(f"\n{'=' * 65}")
print(f"  ✅ Setup complete in {elapsed}s!")
print()
print("  ── NEXT STEP ─────────────────────────────────────────────")
print()
print('  Kernel menu (top right or Kernel -> Change Kernel)')
print(f'  -> Select: "PyTorch 2.0.1 P100 (py3.10)"')
print()
print("  Then run ALL cells from Cell 2 onward.")
print("  (Do NOT use Restart — use Change Kernel)")
print(f"{'=' * 65}")


In [ ]:
# Cell 2 — GPU verification + memory helpers
#
# If you see "No GPU" or kernel errors here:
#   -> Make sure you selected "PyTorch 2.0.1 P100 (py3.10)" as the kernel
#   -> Kaggle: top-right kernel selector, or Kernel -> Change Kernel
#   -> Do NOT restart - CHANGE KERNEL to the one Cell 1 installed
import torch, gc, os
import numpy as np

print(f"Python          : {__import__('sys').version.split()[0]}")
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "\n\nCUDA not available in this kernel.\n"
        "-> Make sure you selected kernel: 'PyTorch 2.0.1 P100 (py3.10)'\n"
        "   (top-right dropdown, or Kernel -> Change Kernel)"
    )

cap = torch.cuda.get_device_capability(0)
print(f"CUDA version    : {torch.version.cuda}")
print(f"GPU             : {torch.cuda.get_device_name(0)}")
print(f"Compute cap     : sm_{cap[0]}{cap[1]}")

# Verify kernels actually run (not just detected)
try:
    _x = torch.randn(128, 128, device="cuda")
    (_x @ _x).sum().item()
    torch.cuda.synchronize()
    del _x
    print("Kernel exec     : ✅ CUDA kernels working")
except Exception as e:
    raise RuntimeError(
        f"CUDA kernels still failing: {e}\n"
        "-> You may still be on the wrong kernel.\n"
        "   Change Kernel -> 'PyTorch 2.0.1 P100 (py3.10)'"
    )

if cap[0] < 7:
    print("AMP             : fp16 (sm_60 does not support TF32 — that is fine)")

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"VRAM            : {vram_gb:.1f} GB")
torch.backends.cudnn.benchmark = True

device      = torch.device("cuda")
DEVICE_NAME = "GPU"


def mem_report(tag=""):
    try:
        import psutil
        ram_used  = psutil.Process(os.getpid()).memory_info().rss / 1e9
        ram_total = psutil.virtual_memory().total / 1e9
    except ImportError:
        ram_used = ram_total = 0.0
    vram_used  = torch.cuda.memory_allocated() / 1e9
    vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"[MEM {tag:<22s}] GPU {vram_used:.2f}/{vram_total:.1f} GB"
          f"  | RAM {ram_used:.2f}/{ram_total:.1f} GB")


def free_memory():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()


mem_report("startup")
print("\n✅ Environment ready — GPU fully operational on sm_60 (P100)")


In [ ]:
# Cell 3 — Dataset auto-detection
from pathlib import Path
from glob    import glob

DATASET_ROOT     = "/kaggle/input/datasets/jaideepchouhan/point-cloud-data-of-10-indian-villages/Training"
DATASET_ZIP_PATH = ""

DATA_SOURCE_MODE = None
TRAINING_ROOT    = None
DATA_ZIP_PATH    = None
ZIP_ROOT_PREFIX  = ""

def has_train_val(root):
    r = Path(root)
    return (r / "train").exists() and (r / "val").exists()

if DATASET_ROOT:
    root = Path(DATASET_ROOT)
    if not has_train_val(root):
        raise FileNotFoundError(f"DATASET_ROOT missing train/val: {root}")
    DATA_SOURCE_MODE, TRAINING_ROOT = "dir", root
elif DATASET_ZIP_PATH:
    DATA_SOURCE_MODE, DATA_ZIP_PATH = "zip", Path(DATASET_ZIP_PATH)
else:
    import zipfile, collections
    dirs = [Path(p) for p in glob("/kaggle/input/**", recursive=True)
            if Path(p).is_dir() and has_train_val(Path(p))]
    if len(dirs) == 1:
        DATA_SOURCE_MODE, TRAINING_ROOT = "dir", dirs[0]
    elif len(dirs) > 1:
        w = [d for d in dirs if (d / "split_manifest.json").exists()]
        if len(w) == 1:
            DATA_SOURCE_MODE, TRAINING_ROOT = "dir", w[0]
    if DATA_SOURCE_MODE is None:
        zips = sorted(glob("/kaggle/input/**/*.zip", recursive=True))
        if len(zips) == 1:
            DATA_SOURCE_MODE, DATA_ZIP_PATH = "zip", Path(zips[0])

if DATA_SOURCE_MODE == "dir":
    print(f"✅ Extracted dataset : {TRAINING_ROOT}")
elif DATA_SOURCE_MODE == "zip":
    gb = DATA_ZIP_PATH.stat().st_size / 1e9
    print(f"✅ ZIP dataset       : {DATA_ZIP_PATH}  ({gb:.2f} GB)")
else:
    raise ValueError("Cannot auto-detect dataset. Set DATASET_ROOT or DATASET_ZIP_PATH.")


In [ ]:
# Cell 4 — Dataset inspection
import json, zipfile
from collections import Counter

if DATA_SOURCE_MODE == "dir":
    train_root = Path(TRAINING_ROOT) / "train"
    val_root   = Path(TRAINING_ROOT) / "val"
    n_train = len([p for p in train_root.glob("tile_*") if (p / "labels.npy").exists()])
    n_val   = len([p for p in val_root.glob("tile_*")   if (p / "labels.npy").exists()])
    print(f"✅ Train tiles: {n_train}  |  Val tiles: {n_val}")
    first_tile = sorted(train_root.glob("tile_*"))[0]
    sample_pts = np.load(first_tile / "points.npy")
    print(f"   First tile shape : {sample_pts.shape}  dtype: {sample_pts.dtype}")
    del sample_pts
else:
    with zipfile.ZipFile(DATA_ZIP_PATH, "r") as z:
        prefixes, train_set, val_set = Counter(), set(), set()
        for info in z.infolist():
            parts = info.filename.strip("/").split("/")
            for split, s in [("train", train_set), ("val", val_set)]:
                if split in parts:
                    i = parts.index(split)
                    if i + 1 < len(parts) and parts[i+1].startswith("tile_"):
                        s.add(parts[i+1])
                        prefixes["/".join(parts[:i])] += 1
    ZIP_ROOT_PREFIX = prefixes.most_common(1)[0][0] if prefixes else ""
    n_train, n_val = len(train_set), len(val_set)
    print(f"✅ ZIP  Train: {n_train}  |  Val: {n_val}  |  prefix: '{ZIP_ROOT_PREFIX}'")


In [ ]:
# Cell 5 — Configuration (P100 + 19 GB RAM + 10-hour wall-time)
from pathlib import Path

RUN_PROFILE = "full"   # "sanity" | "medium" | "full"

# Timing estimates with 2048 pts/tile on P100 (AMP fp16):
#   sanity : ~2 min    medium : ~39 min    full : ~8.5 h   => fits in 10 h
STAGE_PLAN = [
    {"name": "sanity", "epochs": 10, "max_train_tiles":  200,  "max_val_tiles":  40,  "batch_size": 6},
    {"name": "medium", "epochs": 20, "max_train_tiles": 2000,  "max_val_tiles": 400,  "batch_size": 6},
    {"name": "full",   "epochs": 35, "max_train_tiles": 14418, "max_val_tiles": 3955, "batch_size": 6},
]
PROFILE_ORDER = ["sanity", "medium", "full"]

if RUN_PROFILE not in PROFILE_ORDER:
    raise ValueError(f"Unknown RUN_PROFILE: {RUN_PROFILE}")

start_idx  = PROFILE_ORDER.index(RUN_PROFILE)
stage_plan = STAGE_PLAN[start_idx:]
init_stage = stage_plan[0]

KAGGLE_CONFIG = {
    # Data
    "use_zip_dataset"    : DATA_SOURCE_MODE == "zip",
    "zip_path"           : str(DATA_ZIP_PATH)  if DATA_ZIP_PATH  else "",
    "zip_root_prefix"    : ZIP_ROOT_PREFIX,
    "training_dir"       : str(TRAINING_ROOT)  if TRAINING_ROOT  else None,
    # Feature cache
    "feat_cache_dir"     : "/kaggle/working/feat_cache",
    # Outputs
    "logs_dir"              : "/kaggle/working/logs",
    "model_save_path"       : "/kaggle/working/logs/best_model.pth",
    "threshold_path"        : "/kaggle/working/logs/optimal_threshold.json",
    "history_save_path"     : "/kaggle/working/logs/history.json",
    "curves_save_path"      : "/kaggle/working/logs/training_curves.png",
    "latest_checkpoint_path": "/kaggle/working/logs/latest_checkpoint.pth",
    "resume_checkpoint_path": "",
    # Training
    "resume_training"    : True,
    "epochs"             : init_stage["epochs"],
    # 2048 pts/tile: halves cdist tensor size vs 4096, ~2x faster per batch
    "max_points_per_tile": 2048,
    "random_seed"        : 42,
    # Stage plan
    "stage_plan"                : stage_plan,
    "max_train_tiles"           : init_stage["max_train_tiles"],
    "max_val_tiles"             : init_stage["max_val_tiles"],
    "batch_size"                : init_stage["batch_size"],
    "auto_profile_promotion"    : True,
    "promotion_window"          : 3,
    "promotion_min_val_acc"     : 0.82,
    "promotion_min_f1"          : 0.72,
    "promotion_max_val_acc_std" : 0.02,
    "promotion_min_epochs_in_stage": 3,
    # Optimizer
    "max_lr"             : 0.01,
    "weight_decay"       : 1e-4,
    "grad_clip"          : 5.0,
    # Focal Loss
    "focal_alpha"        : 0.75,
    "focal_gamma"        : 2.0,
    # Early stopping
    "early_stop_patience": 8,
    # DataLoader (19 GB RAM: 2 workers x prefetch 2 = 4 buffered batches)
    "num_workers"        : 2,
    "prefetch_factor"    : 2,
    # AMP: fp16 (P100/sm_60 does not support TF32, but fp16 is fine)
    "use_amp"            : True,
    # Gradient checkpointing: recomputes SA1+SA2 activations on backward
    # Saves ~40% VRAM at ~20% compute cost
    "use_grad_ckpt"      : True,
    # Run validation every N epochs
    "val_every"          : 2,
}

for d in [KAGGLE_CONFIG["logs_dir"], KAGGLE_CONFIG["feat_cache_dir"]]:
    Path(d).mkdir(parents=True, exist_ok=True)

print("✅ Config ready")
print(f"   Profile          : {RUN_PROFILE}")
print(f"   Init stage       : {init_stage['name']}  "
      f"({init_stage['max_train_tiles']} / {init_stage['max_val_tiles']} tiles)")
print(f"   batch_size       : {KAGGLE_CONFIG['batch_size']}")
print(f"   max_points/tile  : {KAGGLE_CONFIG['max_points_per_tile']}  (2048 for 10h budget)")
print(f"   num_workers      : {KAGGLE_CONFIG['num_workers']}")
print(f"   AMP fp16         : {KAGGLE_CONFIG['use_amp']}")
print(f"   Grad checkpointing: {KAGGLE_CONFIG['use_grad_ckpt']}")
print(f"   Focal alpha/gamma: {KAGGLE_CONFIG['focal_alpha']} / {KAGGLE_CONFIG['focal_gamma']}")
mem_report("after config")


In [ ]:
# Cell 6 — Geospatial Feature Cache Builder
# Memory fix vs original: all float32 (was float64 -> halves RAM during build)
# mmap_mode="r" on points.npy: maps file without copying it into RAM
import numpy as np
from pathlib import Path
from tqdm    import tqdm
import time, gc


def compute_geospatial_features(xyz: np.ndarray) -> np.ndarray:
    # Input:  (N, 3) float32
    # Output: (N, 4) float32  [delta_z, roughness, slope, density] z-scored
    xyz   = xyz.astype(np.float32, copy=False)
    x_min = float(xyz[:, 0].min())
    y_min = float(xyz[:, 1].min())
    x_rng = float(xyz[:, 0].max() - x_min) + 1e-6
    y_rng = float(xyz[:, 1].max() - y_min) + 1e-6

    GW = int(np.clip(x_rng / 2.0, 16, 64))
    GH = int(np.clip(y_rng / 2.0, 16, 64))
    NC = GH * GW

    xi = np.clip(((xyz[:, 0] - x_min) / x_rng * GW).astype(np.int32), 0, GW - 1)
    yi = np.clip(((xyz[:, 1] - y_min) / y_rng * GH).astype(np.int32), 0, GH - 1)
    ci = (yi * GW + xi)

    z     = xyz[:, 2].copy()
    c_min = np.full(NC, np.inf,  dtype=np.float32)
    c_sum = np.zeros(NC,         dtype=np.float32)
    c_sq  = np.zeros(NC,         dtype=np.float32)
    c_cnt = np.zeros(NC,         dtype=np.float32)

    np.minimum.at(c_min, ci, z)
    np.add.at(c_sum, ci, z)
    np.add.at(c_sq,  ci, z * z)
    np.add.at(c_cnt, ci, 1.0)

    empty         = (c_cnt == 0)
    z_global_mean = float(z.mean())
    c_cnt[empty]  = 1.0
    c_min[empty]  = z_global_mean
    c_sum[empty]  = z_global_mean
    c_sq[empty]   = z_global_mean ** 2

    c_mean = c_sum / c_cnt
    c_std  = np.sqrt(np.maximum(c_sq / c_cnt - c_mean ** 2, 0.0))

    delta_z   = np.clip(z - c_min[ci], 0.0, None)
    roughness = c_std[ci].copy()

    dtm_grid      = c_min.reshape(GH, GW).astype(np.float32)
    dz_dy, dz_dx  = np.gradient(dtm_grid)
    slope         = np.sqrt(dz_dx ** 2 + dz_dy ** 2).ravel()[ci]
    density       = (c_cnt[ci] / (c_cnt.max() + 1e-6))

    feat = np.stack([delta_z, roughness, slope, density], axis=1).astype(np.float32)
    mu   = feat.mean(axis=0)
    sig  = feat.std(axis=0) + 1e-6
    feat = (feat - mu) / sig

    del c_min, c_sum, c_sq, c_cnt, c_mean, c_std, dtm_grid, dz_dx, dz_dy
    return feat


def build_feature_cache(cfg: dict, force_rebuild: bool = False) -> None:
    if cfg.get("use_zip_dataset", False):
        print("⚠️  ZIP mode: features will be computed on-the-fly (no cache supported).")
        return

    training_root  = Path(cfg["training_dir"])
    feat_cache_dir = Path(cfg["feat_cache_dir"])
    feat_cache_dir.mkdir(parents=True, exist_ok=True)

    total_built = total_skipped = total_errors = 0
    t_global = time.time()

    for split in ("train", "val"):
        split_root      = training_root / split
        split_cache_dir = feat_cache_dir / split
        split_cache_dir.mkdir(exist_ok=True)
        if not split_root.exists():
            continue
        tiles = sorted([
            p.name for p in split_root.glob("tile_*")
            if (p / "labels.npy").exists()
        ])
        print(f"\n📐 [{split}] {len(tiles)} tiles")
        t0 = time.time()
        for k, tile in enumerate(tqdm(tiles, desc=f"  {split}", ncols=80)):
            cache_path = split_cache_dir / f"{tile}.npy"
            if cache_path.exists() and not force_rebuild:
                total_skipped += 1
                continue
            try:
                # mmap_mode="r": maps file into virtual memory without full RAM copy
                xyz  = np.load(split_root / tile / "points.npy",
                               mmap_mode="r")[:, :3].astype(np.float32)
                feat = compute_geospatial_features(xyz)
                np.save(cache_path, feat)
                total_built += 1
                del xyz, feat
            except Exception as e:
                print(f"  ⚠️  {tile}: {e}")
                total_errors += 1
            if k % 1000 == 0 and k > 0:
                gc.collect()  # prevent gradual RAM accumulation
        gc.collect()
        print(f"  Done in {time.time() - t0:.0f}s")

    print(f"\n✅ Cache: {total_built} built  {total_skipped} cached  {total_errors} errors")
    print(f"   Total time : {time.time() - t_global:.0f}s")
    mem_report("after cache build")


build_feature_cache(KAGGLE_CONFIG, force_rebuild=False)


In [ ]:
# Cell 7 — Enhanced PointNet++ with MSG + Gradient Checkpointing
# All torch.cuda.amp.* APIs: compatible with PyTorch 2.0.1 on P100 (sm_60).
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as grad_checkpoint


def farthest_point_sample(xyz: torch.Tensor, n_sample: int) -> torch.Tensor:
    B, N, _ = xyz.shape
    dev  = xyz.device
    cent = torch.zeros(B, n_sample, dtype=torch.long,  device=dev)
    dist = torch.full((B, N), 1e10, dtype=torch.float32, device=dev)
    far  = torch.randint(0, N, (B,), dtype=torch.long, device=dev)
    bidx = torch.arange(B, dtype=torch.long, device=dev)
    for i in range(n_sample):
        cent[:, i] = far
        c    = xyz[bidx, far, :].unsqueeze(1)
        d    = ((xyz - c) ** 2).sum(dim=-1)
        dist = torch.where(d < dist, d, dist)
        far  = dist.argmax(dim=-1)
    return cent


def index_points(pts: torch.Tensor, idx: torch.Tensor) -> torch.Tensor:
    B    = pts.shape[0]
    shp  = [B] + [1] * (idx.dim() - 1)
    bidx = torch.arange(B, device=pts.device).view(shp).expand_as(idx)
    return pts[bidx, idx]


class SetAbstractionMSG(nn.Module):
    # Multi-Scale Grouping SA.
    # Computes cdist once and reuses it for all radius scales (saves VRAM).
    def __init__(self, n_ctr, radii, samples, in_ch, mlp_specs):
        super().__init__()
        self.n_ctr   = n_ctr
        self.radii   = radii
        self.samples = samples
        self.mlps    = nn.ModuleList()
        for mlp_dims in mlp_specs:
            layers, last = [], in_ch + 3
            for d in mlp_dims:
                layers += [nn.Conv2d(last, d, 1, bias=False),
                           nn.BatchNorm2d(d), nn.ReLU(inplace=True)]
                last = d
            self.mlps.append(nn.Sequential(*layers))

    def forward(self, xyz, points):
        fps_idx = farthest_point_sample(xyz, self.n_ctr)
        new_xyz = index_points(xyz, fps_idx)           # (B, M, 3)
        dist    = torch.cdist(new_xyz, xyz)            # (B, M, N) -- computed once, reused

        outs = []
        for r, k, mlp in zip(self.radii, self.samples, self.mlps):
            masked  = torch.where(dist <= r, dist, dist.new_full((), 1e10))
            idx     = masked.topk(k, dim=-1, largest=False)[1]
            grp_xyz = index_points(xyz, idx) - new_xyz.unsqueeze(2)
            grp_pts = (torch.cat([grp_xyz, index_points(points, idx)], dim=-1)
                       if points is not None else grp_xyz)
            feat    = mlp(grp_pts.permute(0, 3, 2, 1)).max(dim=2)[0]
            outs.append(feat.permute(0, 2, 1))
            del grp_xyz, grp_pts, feat, idx, masked  # free each scale immediately

        del dist
        return new_xyz, torch.cat(outs, dim=-1)


class SetAbstraction(nn.Module):
    # Single-scale SA for the global context layer (SA3).
    def __init__(self, n_ctr, radius, n_samp, in_ch, mlp_dims):
        super().__init__()
        self.n_ctr  = n_ctr
        self.radius = radius
        self.n_samp = n_samp
        layers, last = [], in_ch + 3
        for d in mlp_dims:
            layers += [nn.Conv2d(last, d, 1, bias=False),
                       nn.BatchNorm2d(d), nn.ReLU(inplace=True)]
            last = d
        self.mlp = nn.Sequential(*layers)

    def forward(self, xyz, points):
        fps_idx = farthest_point_sample(xyz, self.n_ctr)
        new_xyz = index_points(xyz, fps_idx)
        dist    = torch.cdist(new_xyz, xyz)
        masked  = torch.where(dist <= self.radius, dist, dist.new_full((), 1e10))
        idx     = masked.topk(self.n_samp, dim=-1, largest=False)[1]
        grp_xyz = index_points(xyz, idx) - new_xyz.unsqueeze(2)
        grp_pts = (torch.cat([grp_xyz, index_points(points, idx)], dim=-1)
                   if points is not None else grp_xyz)
        feat    = self.mlp(grp_pts.permute(0, 3, 2, 1)).max(dim=2)[0].permute(0, 2, 1)
        del dist, masked, grp_xyz, grp_pts
        return new_xyz, feat


class FeaturePropagation(nn.Module):
    def __init__(self, in_ch, mlp_dims):
        super().__init__()
        layers, last = [], in_ch
        for d in mlp_dims:
            layers += [nn.Conv1d(last, d, 1, bias=False),
                       nn.BatchNorm1d(d), nn.ReLU(inplace=True)]
            last = d
        self.mlp = nn.Sequential(*layers)

    def forward(self, xyz1, xyz2, pts1, pts2):
        dists, idx = torch.cdist(xyz1, xyz2).topk(3, dim=-1, largest=False)
        dists      = torch.clamp(dists, min=1e-10)
        w          = 1.0 / dists
        w          = w / w.sum(dim=-1, keepdim=True)
        interp     = (index_points(pts2, idx) * w.unsqueeze(-1)).sum(dim=2)
        feat = torch.cat([pts1, interp], dim=-1) if pts1 is not None else interp
        return self.mlp(feat.permute(0, 2, 1)).permute(0, 2, 1)


class DTMPointNet2(nn.Module):
    # Enhanced PointNet++ with MSG for DTM ground segmentation.
    # Input  : (B, N, 7) = [x, y, z, delta_z, roughness, slope, density]
    # Output : (B, N, 2) = [non-ground logit, ground logit]
    # use_grad_ckpt: recomputes SA1+SA2 activations on backward (~40% VRAM saving)
    IN_FEAT_CH = 4

    def __init__(self, num_classes=2, use_grad_ckpt=False):
        super().__init__()
        self.use_grad_ckpt = use_grad_ckpt
        IC = self.IN_FEAT_CH

        self.sa1 = SetAbstractionMSG(
            n_ctr=512, radii=[0.5, 1.5], samples=[32, 64], in_ch=IC,
            mlp_specs=[[32, 64], [64, 128]]
        )
        self.sa2 = SetAbstractionMSG(
            n_ctr=128, radii=[3.0, 6.0], samples=[64, 128], in_ch=192,
            mlp_specs=[[128, 128], [128, 256]]
        )
        self.sa3 = SetAbstraction(
            n_ctr=32, radius=12.0, n_samp=128, in_ch=384,
            mlp_dims=[256, 512]
        )
        self.fp3 = FeaturePropagation(384 + 512, [512, 256])
        self.fp2 = FeaturePropagation(192 + 256, [256, 128])
        self.fp1 = FeaturePropagation(IC  + 128, [128, 128])
        self.head = nn.Sequential(
            nn.Conv1d(128, 128, 1, bias=False), nn.BatchNorm1d(128), nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Conv1d(128,  64, 1, bias=False), nn.BatchNorm1d(64),  nn.ReLU(inplace=True),
            nn.Conv1d( 64, num_classes, 1),
        )

    def _sa1_fn(self, xyz, pts):
        return self.sa1(xyz, pts)

    def _sa2_fn(self, xyz, pts):
        return self.sa2(xyz, pts)

    def forward(self, x):
        xyz    = x[:, :, :3].contiguous()
        l0_pts = x[:, :, 3:].contiguous()

        if self.use_grad_ckpt and self.training:
            # use_reentrant=False: available since PyTorch 1.12 -- compatible with 2.0.1
            l1_xyz, l1_pts = grad_checkpoint(self._sa1_fn, xyz,    l0_pts, use_reentrant=False)
            l2_xyz, l2_pts = grad_checkpoint(self._sa2_fn, l1_xyz, l1_pts, use_reentrant=False)
        else:
            l1_xyz, l1_pts = self.sa1(xyz,    l0_pts)
            l2_xyz, l2_pts = self.sa2(l1_xyz, l1_pts)

        l3_xyz, l3_pts = self.sa3(l2_xyz, l2_pts)
        l2_pts = self.fp3(l2_xyz, l3_xyz, l2_pts, l3_pts)
        l1_pts = self.fp2(l1_xyz, l2_xyz, l1_pts, l2_pts)
        l0_pts = self.fp1(xyz,    l1_xyz, l0_pts, l1_pts)
        return self.head(l0_pts.permute(0, 2, 1)).permute(0, 2, 1)


class FocalLoss(nn.Module):
    # Focal Loss: fixes Precision/Recall imbalance.
    # alpha=0.75 upweights minority ground class; gamma focuses on hard points.
    def __init__(self, alpha: float = 0.75, gamma: float = 2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        ce      = F.cross_entropy(logits, targets, reduction="none")
        pt      = torch.exp(-ce)
        alpha_t = torch.where(targets == 1,
                              logits.new_full(ce.shape, self.alpha),
                              logits.new_full(ce.shape, 1.0 - self.alpha))
        return (alpha_t * (1.0 - pt) ** self.gamma * ce).mean()


def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)


# Smoke-test: forward pass with tiny batch
_m = DTMPointNet2(use_grad_ckpt=True).to(device)
with torch.no_grad():
    _x = torch.randn(2, 64, 7, device=device)
    _y = _m(_x)
assert _y.shape == (2, 64, 2), f"Unexpected output shape: {_y.shape}"
print(f"✅ DTMPointNet2 smoke-test passed")
print(f"   Parameters : {count_params(_m):,}")
print(f"   Output     : {_y.shape}")
del _m, _x, _y
free_memory()
mem_report("after model def")


In [ ]:
# Cell 8 — Dataset class (mmap loading + cached or on-the-fly features)
import io, zipfile
import numpy as np
from pathlib import Path, PurePosixPath
import torch
from torch.utils.data import Dataset


class DTMPointCloudDataset(Dataset):
    # Loads LiDAR tiles and returns (N, 7) feature tensors.
    # Features: [x, y, z, delta_z, roughness, slope, density]
    # Memory notes:
    #   - float32 cache on disk (no float64 intermediate in __getitem__)
    #   - mmap_mode="r" on .npy: maps file without full RAM copy
    #   - ZIP mode: streams bytes directly, no extraction to disk

    def __init__(
        self,
        root_dir=None, zip_path=None, zip_root_prefix="",
        feat_cache_dir=None, split="train", augment=False,
        max_points=2048, max_tiles=None, seed=42,
    ):
        self.augment        = augment
        self.max_points     = max_points
        self.seed           = seed
        self.split          = split
        self.mode           = "zip" if zip_path is not None else "dir"
        self.feat_cache_dir = Path(feat_cache_dir) / split if feat_cache_dir else None
        self._zip           = None

        if self.mode == "dir":
            self.root_dir = Path(root_dir)
            if self.root_dir.name in {"train", "val"}:
                self.split = self.root_dir.name
            self.tiles = [
                p.name for p in self.root_dir.glob("tile_*")
                if (p / "labels.npy").exists()
            ]
        else:
            self.root_dir        = None
            self.zip_path        = Path(zip_path)
            self.zip_root_prefix = zip_root_prefix.strip("/")
            self.tiles           = self._discover_tiles_zip()

        self.tiles = sorted(self.tiles)
        if max_tiles is not None and len(self.tiles) > max_tiles:
            rng  = np.random.default_rng(seed)
            pick = rng.choice(len(self.tiles), size=max_tiles, replace=False)
            self.tiles = [self.tiles[i] for i in sorted(pick)]

        self._use_cache = False
        if self.mode == "dir" and self.feat_cache_dir is not None and self.tiles:
            if self.feat_cache_dir.exists():
                self._use_cache = (self.feat_cache_dir / f"{self.tiles[0]}.npy").exists()

        status = "✅ float32 cache" if self._use_cache else "⚠️  on-the-fly"
        print(f"[{self.split}] {len(self.tiles)} tiles  |  features: {status}")

    def _discover_tiles_zip(self):
        tiles = set()
        with zipfile.ZipFile(self.zip_path, "r") as zf:
            for info in zf.infolist():
                parts = PurePosixPath(info.filename).parts
                if self.split in parts:
                    i = parts.index(self.split)
                    if i + 1 < len(parts) and parts[i+1].startswith("tile_"):
                        tiles.add(parts[i+1])
        return list(tiles)

    def _zip_member(self, tile, fname):
        pre = f"{self.zip_root_prefix}/" if self.zip_root_prefix else ""
        return f"{pre}{self.split}/{tile}/{fname}"

    def _ensure_zip(self):
        if self._zip is None:
            self._zip = zipfile.ZipFile(self.zip_path, "r")

    def _load_npy_zip(self, member):
        self._ensure_zip()
        with self._zip.open(member) as f:
            return np.load(io.BytesIO(f.read()))

    def load_labels(self, idx):
        tile = self.tiles[idx]
        if self.mode == "dir":
            return np.load(self.root_dir / tile / "labels.npy").astype(np.int64)
        return self._load_npy_zip(self._zip_member(tile, "labels.npy")).astype(np.int64)

    def _load_xyz_labels(self, tile):
        if self.mode == "dir":
            # mmap_mode="r": maps .npy into virtual address space, no RAM copy
            xyz    = np.load(self.root_dir / tile / "points.npy",
                             mmap_mode="r")[:, :3].astype(np.float32)
            labels = np.load(self.root_dir / tile / "labels.npy").astype(np.int64)
        else:
            xyz    = self._load_npy_zip(
                         self._zip_member(tile, "points.npy")).astype(np.float32)[:, :3]
            labels = self._load_npy_zip(
                         self._zip_member(tile, "labels.npy")).astype(np.int64)
        return xyz, labels

    def __len__(self):
        return len(self.tiles)

    def __getitem__(self, idx):
        tile        = self.tiles[idx]
        xyz, labels = self._load_xyz_labels(tile)

        if self._use_cache:
            feat_extra = np.load(self.feat_cache_dir / f"{tile}.npy")  # (N_orig, 4) float32
        else:
            feat_extra = compute_geospatial_features(xyz)               # on-the-fly float32

        N      = xyz.shape[0]
        choice = np.random.choice(N, self.max_points, replace=(N < self.max_points))
        xyz        = xyz[choice]
        feat_extra = feat_extra[choice]
        labels     = labels[choice]

        if self.augment:
            angle = np.random.uniform(0.0, 2.0 * np.pi)
            c, s  = np.cos(angle).astype(np.float32), np.sin(angle).astype(np.float32)
            R     = np.array([[c, -s, 0.0], [s, c, 0.0], [0.0, 0.0, 1.0]], dtype=np.float32)
            xyz   = xyz @ R.T
            xyz  += np.random.normal(0, 0.02, xyz.shape).astype(np.float32)
            xyz  *= np.float32(np.random.uniform(0.90, 1.10))
            if np.random.rand() > 0.5:
                xyz[:, 0] *= -1.0
            if np.random.rand() > 0.5:
                xyz[:, 1] *= -1.0
            sx, sy = np.random.uniform(0.85, 1.15, 2).astype(np.float32)
            xyz[:, 0] *= sx
            xyz[:, 1] *= sy
            xyz[:, 2] *= np.float32(np.random.uniform(0.80, 1.20))
            if np.random.rand() < 0.3:
                keep_n  = int(self.max_points * np.random.uniform(0.75, 1.00))
                keep    = np.random.choice(self.max_points, keep_n, replace=False)
                refill  = np.random.choice(keep, self.max_points - keep_n, replace=True)
                order   = np.concatenate([keep, refill])
                xyz        = xyz[order]
                feat_extra = feat_extra[order]
                labels     = labels[order]

        xyz[:, 0] -= xyz[:, 0].mean()
        xyz[:, 1] -= xyz[:, 1].mean()
        features = np.concatenate([xyz, feat_extra], axis=1).astype(np.float32)
        return torch.from_numpy(features), torch.from_numpy(labels)

    def __del__(self):
        if getattr(self, "_zip", None) is not None:
            try:
                self._zip.close()
            except Exception:
                pass


In [ ]:
# Cell 9 — Training loop
# Key compatibility fix: torch.cuda.amp.GradScaler / torch.cuda.amp.autocast
# (NOT torch.amp.* which is 2.0+ only, but torch.cuda.amp.* works on 2.0.1)
import time, json
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm


def build_loaders(cfg, device_name):
    common = dict(
        feat_cache_dir  = cfg["feat_cache_dir"],
        max_points      = cfg["max_points_per_tile"],
        seed            = cfg["random_seed"],
    )
    if cfg["use_zip_dataset"]:
        zk = dict(zip_path=cfg["zip_path"],
                  zip_root_prefix=cfg.get("zip_root_prefix", ""))
        tr_ds = DTMPointCloudDataset(**zk, **common, split="train",
                                     augment=True,  max_tiles=cfg["max_train_tiles"])
        va_ds = DTMPointCloudDataset(**zk, **common, split="val",
                                     augment=False, max_tiles=cfg["max_val_tiles"])
    else:
        tdir  = cfg["training_dir"]
        tr_ds = DTMPointCloudDataset(root_dir=tdir + "/train", **common,
                                     augment=True,  max_tiles=cfg["max_train_tiles"])
        va_ds = DTMPointCloudDataset(root_dir=tdir + "/val",   **common,
                                     augment=False, max_tiles=cfg["max_val_tiles"])

    pin = (device_name == "GPU")
    nw  = cfg.get("num_workers", 2)
    pf  = cfg.get("prefetch_factor", 2)
    kw  = dict(
        num_workers       = nw,
        pin_memory        = pin,
        persistent_workers= (nw > 0),
        prefetch_factor   = (pf if nw > 0 else None),
    )
    tr_ld = DataLoader(tr_ds, batch_size=cfg["batch_size"],
                       shuffle=True, drop_last=True,  **kw)
    va_ld = DataLoader(va_ds, batch_size=cfg["batch_size"],
                       shuffle=False, drop_last=False, **kw)
    return tr_ds, va_ds, tr_ld, va_ld


def should_promote_stage(history, stage_name, cfg):
    window     = int(cfg.get("promotion_window", 3))
    min_epochs = int(cfg.get("promotion_min_epochs_in_stage", 3))
    sh = [h for h in history
          if h.get("stage") == stage_name
          and not np.isnan(h.get("va", float("nan")))]
    if len(sh) < max(window, min_epochs):
        return False
    recent = sh[-window:]
    vals   = np.array([h["va"] for h in recent])
    f1s    = np.array([h["f1"] for h in recent])
    return (vals.mean() >= cfg.get("promotion_min_val_acc", 0.82)
            and f1s.mean() >= cfg.get("promotion_min_f1", 0.72)
            and vals.std()  <= cfg.get("promotion_max_val_acc_std", 0.02))


def apply_stage(cfg, stage_plan, idx):
    s = stage_plan[idx]
    cfg["max_train_tiles"] = int(s["max_train_tiles"])
    cfg["max_val_tiles"]   = int(s["max_val_tiles"])
    cfg["batch_size"]      = int(s.get("batch_size", cfg.get("batch_size", 6)))
    cfg["epochs"]          = int(s.get("epochs", cfg.get("epochs", 35)))
    return s["name"]


def save_curves(history, path):
    val_h   = [h for h in history if not np.isnan(h.get("vl", float("nan")))]
    eps_all = [h["ep"] for h in history]
    eps_val = [h["ep"] for h in val_h]
    fig, ax = plt.subplots(1, 3, figsize=(16, 4))
    ax[0].plot(eps_all, [h["tl"] for h in history], label="Train")
    if val_h:
        ax[0].plot(eps_val, [h["vl"] for h in val_h], label="Val")
    ax[0].set(title="Loss", xlabel="Epoch"); ax[0].legend(); ax[0].grid(True)
    ax[1].plot(eps_all, [h["ta"] for h in history], label="Train acc")
    if val_h:
        ax[1].plot(eps_val, [h["va"] for h in val_h], label="Val acc")
    ax[1].axhline(0.95, ls="--", color="red", alpha=0.4, label="95% target")
    ax[1].set(title="Accuracy", xlabel="Epoch", ylim=(0.8, 1.0))
    ax[1].legend(); ax[1].grid(True)
    if val_h:
        ax[2].plot(eps_val, [h["p"]  for h in val_h], label="Precision")
        ax[2].plot(eps_val, [h["r"]  for h in val_h], label="Recall")
        ax[2].plot(eps_val, [h["f1"] for h in val_h], label="F1", lw=2)
    ax[2].set(title="P / R / F1", xlabel="Epoch"); ax[2].legend(); ax[2].grid(True)
    plt.suptitle("DTMPointNet++ -- Kaggle P100 (PyTorch 2.0.1+cu117)", fontsize=13)
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show(); plt.close(fig)


def train(cfg, device, device_name):
    torch.manual_seed(cfg["random_seed"])
    np.random.seed(cfg["random_seed"])

    model = DTMPointNet2(
        num_classes   = 2,
        use_grad_ckpt = cfg.get("use_grad_ckpt", True),
    ).to(device)
    print(f"Model parameters: {count_params(model):,}")

    criterion = FocalLoss(alpha=cfg["focal_alpha"], gamma=cfg["focal_gamma"])
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr           = cfg["max_lr"] / 25.0,
        weight_decay = cfg["weight_decay"],
        betas        = (0.9, 0.999),
    )

    # torch.cuda.amp.GradScaler -- the safe API for PyTorch 2.0.1 / sm_60
    use_amp = cfg.get("use_amp", True) and device_name == "GPU"
    scaler  = torch.cuda.amp.GradScaler(enabled=use_amp)

    stage_plan        = cfg.get("stage_plan", [])
    current_stage_idx = 0
    start_epoch       = 1
    best_val_acc      = 0.0
    patience_ctr      = 0
    history           = []

    ckpt_path = Path(cfg.get("resume_checkpoint_path", "").strip()
                     or cfg.get("latest_checkpoint_path", ""))
    if cfg.get("resume_training", True) and ckpt_path.exists():
        ckpt = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ckpt["model_state_dict"])
        if "optimizer_state_dict" in ckpt:
            optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        if "scaler_state_dict" in ckpt:
            scaler.load_state_dict(ckpt["scaler_state_dict"])
        start_epoch       = ckpt.get("epoch", 0) + 1
        best_val_acc      = ckpt.get("best_val_acc", 0.0)
        patience_ctr      = ckpt.get("patience_ctr", 0)
        history           = ckpt.get("history", [])
        current_stage_idx = ckpt.get("current_stage_idx", 0)
        apply_stage(cfg, stage_plan, current_stage_idx)
        print(f"✅ Resumed: epoch {start_epoch}, best_val={best_val_acc:.4f}")
    else:
        apply_stage(cfg, stage_plan, 0)
        print("ℹ️  Starting fresh")

    stage_name = stage_plan[current_stage_idx]["name"]
    tr_ds, va_ds, tr_ld, va_ld = build_loaders(cfg, device_name)

    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr          = cfg["max_lr"],
        epochs          = cfg["epochs"],
        steps_per_epoch = len(tr_ld),
        pct_start       = 0.30,
        anneal_strategy = "cos",
        div_factor      = 25.0,
        final_div_factor= 1000.0,
    )
    if cfg.get("resume_training", True) and ckpt_path.exists():
        ckpt2 = torch.load(ckpt_path, map_location=device)
        if "scheduler_state_dict" in ckpt2:
            try:
                scheduler.load_state_dict(ckpt2["scheduler_state_dict"])
            except Exception:
                pass  # shape mismatch after stage change is OK

    val_every = max(1, cfg.get("val_every", 2))
    print(f"\n▶  Stage: {stage_name} | AMP={use_amp} | "
          f"GradCkpt={cfg.get('use_grad_ckpt', True)} | val_every={val_every}")
    print(f"   Batches: train={len(tr_ld)}  val={len(va_ld)}")
    mem_report("training start")

    for epoch in range(start_epoch, cfg["epochs"] + 1):
        t0 = time.time()

        # ── Train ──────────────────────────────────────────────────────────
        model.train()
        tr_loss = tr_ok = tr_n = 0

        for feat, labels in tqdm(tr_ld, desc=f"Ep {epoch:02d} Train",
                                  leave=False, ncols=80):
            feat   = feat.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)

            # torch.cuda.amp.autocast -- compatible with PyTorch 2.0.1 on P100
            with torch.cuda.amp.autocast(enabled=use_amp):
                logits  = model(feat)
                B, N, C = logits.shape
                loss    = criterion(logits.reshape(B * N, C),
                                    labels.reshape(B * N))

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            with torch.no_grad():
                preds   = logits.argmax(-1)
                tr_ok  += (preds == labels).sum().item()
                tr_n   += B * N
                tr_loss += loss.item() * B

        t_loss = tr_loss / max(1, len(tr_ds))
        t_acc  = tr_ok   / max(1, tr_n)
        cur_lr = scheduler.get_last_lr()[0]

        # ── Validate ────────────────────────────────────────────────────────
        run_val = (epoch % val_every == 0) or (epoch == cfg["epochs"])
        v_loss = v_acc = prec = rec = f1 = float("nan")

        if run_val:
            model.eval()
            va_ls = va_ok = va_n = va_tp = va_fp = va_fn = 0
            with torch.no_grad():
                for feat, labels in tqdm(va_ld, desc=f"Ep {epoch:02d} Val",
                                          leave=False, ncols=80):
                    feat   = feat.to(device, non_blocking=True)
                    labels = labels.to(device, non_blocking=True)
                    with torch.cuda.amp.autocast(enabled=use_amp):
                        logits  = model(feat)
                        B, N, C = logits.shape
                        loss    = criterion(logits.reshape(B * N, C),
                                            labels.reshape(B * N))
                    preds   = logits.argmax(-1)
                    va_ls  += loss.item() * B
                    va_ok  += (preds == labels).sum().item()
                    va_n   += B * N
                    va_tp  += ((preds == 1) & (labels == 1)).sum().item()
                    va_fp  += ((preds == 1) & (labels == 0)).sum().item()
                    va_fn  += ((preds == 0) & (labels == 1)).sum().item()

            v_loss = va_ls / max(1, len(va_ds))
            v_acc  = va_ok / max(1, va_n)
            prec   = va_tp / (va_tp + va_fp + 1e-9)
            rec    = va_tp / (va_tp + va_fn + 1e-9)
            f1     = 2 * prec * rec / (prec + rec + 1e-9)
            free_memory()  # release VRAM after every validation pass

        elapsed = time.time() - t0

        if run_val:
            flag = "  *** BEST ***" if v_acc > best_val_acc else ""
            print(f"Ep {epoch:3d}/{cfg['epochs']} [{stage_name}] "
                  f"| T {t_loss:.4f}/{t_acc:.4f} "
                  f"| V {v_loss:.4f}/{v_acc:.4f} "
                  f"| P {prec:.3f} R {rec:.3f} F1 {f1:.3f} "
                  f"| lr {cur_lr:.2e} {elapsed:.0f}s{flag}")
        else:
            print(f"Ep {epoch:3d}/{cfg['epochs']} [{stage_name}] "
                  f"| T {t_loss:.4f}/{t_acc:.4f} | val skip "
                  f"| lr {cur_lr:.2e} {elapsed:.0f}s")

        history.append(dict(
            ep=epoch, stage=stage_name,
            tl=t_loss, ta=t_acc,
            vl=v_loss,
            va=v_acc  if run_val else (history[-1]["va"] if history else 0.0),
            p=prec    if run_val else float("nan"),
            r=rec     if run_val else float("nan"),
            f1=f1     if run_val else (history[-1]["f1"] if history else 0.0),
            lr=cur_lr,
        ))

        if run_val:
            if v_acc > best_val_acc:
                best_val_acc = v_acc
                patience_ctr = 0
                torch.save(model.state_dict(), cfg["model_save_path"])
            else:
                patience_ctr += 1

        torch.save({
            "epoch"               : epoch,
            "best_val_acc"        : best_val_acc,
            "patience_ctr"        : patience_ctr,
            "current_stage_idx"   : current_stage_idx,
            "model_state_dict"    : model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict"   : scaler.state_dict(),
            "history"             : history,
            "cfg"                 : cfg,
        }, cfg["latest_checkpoint_path"])

        # Auto-stage promotion
        if (run_val and cfg.get("auto_profile_promotion", True)
                and current_stage_idx + 1 < len(stage_plan)):
            if should_promote_stage(history, stage_name, cfg):
                current_stage_idx += 1
                stage_name = apply_stage(cfg, stage_plan, current_stage_idx)
                print(f"\n🚀 Promoted -> '{stage_name}'  "
                      f"({cfg['max_train_tiles']}/{cfg['max_val_tiles']} tiles, "
                      f"batch={cfg['batch_size']})")
                free_memory()
                tr_ds, va_ds, tr_ld, va_ld = build_loaders(cfg, device_name)
                scheduler = torch.optim.lr_scheduler.OneCycleLR(
                    optimizer,
                    max_lr          = cfg["max_lr"],
                    epochs          = cfg["epochs"] - epoch,
                    steps_per_epoch = len(tr_ld),
                    pct_start       = 0.10,
                    anneal_strategy = "cos",
                    div_factor      = 10.0,
                    final_div_factor= 1000.0,
                )
                patience_ctr = 0
                mem_report(f"stage->{stage_name}")

        if run_val and patience_ctr >= cfg["early_stop_patience"]:
            print(f"\n  Early stopping at epoch {epoch} "
                  f"(no improvement for {patience_ctr} val checks)")
            break

    with open(cfg["history_save_path"], "w") as f:
        json.dump(history, f, indent=2)
    save_curves(history, cfg["curves_save_path"])

    print(f"\n✅ Training complete.  Best val acc : {best_val_acc:.4f}")
    free_memory()
    mem_report("done")
    return model, history, best_val_acc


# ── Run ────────────────────────────────────────────────────────────────────────
trained_model, train_history, best_acc = train(KAGGLE_CONFIG, device, DEVICE_NAME)


In [ ]:
# Cell 10 — Post-training threshold sweep + final evaluation
import json
import numpy as np
import torch
import torch.nn.functional as F
from tqdm import tqdm
from pathlib import Path
import matplotlib.pyplot as plt


def find_optimal_threshold(model, val_loader, device, use_amp=True,
                            thresholds=None):
    if thresholds is None:
        thresholds = np.arange(0.20, 0.85, 0.01)
    model.eval()
    all_probs, all_labels = [], []
    print("Threshold search -- full val pass...")
    with torch.no_grad():
        for feat, labels in tqdm(val_loader, ncols=80):
            feat = feat.to(device, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=(use_amp and device.type == "cuda")):
                logits = model(feat)
            probs = F.softmax(logits, dim=-1)[:, :, 1]
            all_probs.append( probs.reshape(-1).float().cpu().numpy())
            all_labels.append(labels.reshape(-1).numpy())
    free_memory()

    all_probs  = np.concatenate(all_probs).astype(np.float32)
    all_labels = np.concatenate(all_labels).astype(np.int32)

    best_thresh, best_f1, best_row = 0.5, 0.0, {}
    rows = []
    for t in thresholds:
        preds = (all_probs >= t).astype(np.int32)
        tp  = int(((preds == 1) & (all_labels == 1)).sum())
        fp  = int(((preds == 1) & (all_labels == 0)).sum())
        fn  = int(((preds == 0) & (all_labels == 1)).sum())
        tn  = int(((preds == 0) & (all_labels == 0)).sum())
        p   = tp / (tp + fp + 1e-9)
        r   = tp / (tp + fn + 1e-9)
        f1  = 2 * p * r / (p + r + 1e-9)
        acc = (tp + tn) / (len(all_labels) + 1e-9)
        rows.append(dict(t=round(float(t), 2), acc=acc, p=p, r=r, f1=f1))
        if f1 > best_f1:
            best_f1, best_thresh, best_row = f1, float(t), rows[-1]

    print(f"\n🎯 Optimal threshold : {best_thresh:.2f}")
    print(f"   Accuracy          : {best_row['acc']:.4f}")
    print(f"   Precision         : {best_row['p']:.4f}")
    print(f"   Recall            : {best_row['r']:.4f}")
    print(f"   F1 (ground)       : {best_row['f1']:.4f}")

    fig, ax = plt.subplots(figsize=(9, 4))
    ts  = [r["t"]   for r in rows]
    accs = [r["acc"] for r in rows]
    f1s  = [r["f1"]  for r in rows]
    ax.plot(ts, accs, label="Accuracy", lw=2)
    ax.plot(ts, f1s,  label="F1 (ground)", lw=2)
    ax.axvline(best_thresh, ls="--", color="red",  label=f"Best={best_thresh:.2f}")
    ax.axhline(0.95,         ls=":",  color="gray", alpha=0.5, label="95% target")
    ax.set(title="Threshold vs Accuracy / F1", xlabel="Threshold")
    ax.legend(); ax.grid(True)
    plt.tight_layout()
    plt.savefig(KAGGLE_CONFIG["logs_dir"] + "/threshold_curve.png", dpi=150)
    plt.show(); plt.close(fig)
    return best_thresh, best_row, rows


# Load best model (use_grad_ckpt=False: no overhead at eval time)
best_model = DTMPointNet2(num_classes=2, use_grad_ckpt=False).to(device)
best_model.load_state_dict(
    torch.load(KAGGLE_CONFIG["model_save_path"], map_location=device))
print("✅ Best model weights loaded")

_, _, _, val_ld_eval = build_loaders(KAGGLE_CONFIG, DEVICE_NAME)

opt_thresh, opt_metrics, all_rows = find_optimal_threshold(
    best_model, val_ld_eval, device, use_amp=KAGGLE_CONFIG["use_amp"])

with open(KAGGLE_CONFIG["threshold_path"], "w") as f:
    json.dump({"optimal_threshold": opt_thresh,
               "metrics_at_threshold": opt_metrics,
               "sweep": all_rows}, f, indent=2)

print(f"\n💾 Saved -> {KAGGLE_CONFIG['threshold_path']}")
print("\n" + "=" * 60)
print("  FINAL RESULTS SUMMARY")
print("=" * 60)
print(f"  Best val acc (thresh=0.50)      : {best_acc:.4f}")
print(f"  Accuracy at thresh={opt_thresh:.2f}          : {opt_metrics['acc']:.4f}")
print(f"  Precision                       : {opt_metrics['p']:.4f}")
print(f"  Recall                          : {opt_metrics['r']:.4f}")
print(f"  F1 (ground class)               : {opt_metrics['f1']:.4f}")
met = opt_metrics["acc"] >= 0.95
print(f"\n  95% target met: {'✅ YES' if met else '❌ Not yet -- extend epochs or lower threshold'}")
print("=" * 60)
mem_report("final")


In [ ]:
# Cell 11 — Collect and zip all outputs
from pathlib import Path
import zipfile

outputs = [
    KAGGLE_CONFIG["model_save_path"],
    KAGGLE_CONFIG["latest_checkpoint_path"],
    KAGGLE_CONFIG["history_save_path"],
    KAGGLE_CONFIG["curves_save_path"],
    KAGGLE_CONFIG["threshold_path"],
    KAGGLE_CONFIG["logs_dir"] + "/threshold_curve.png",
]

missing = [p for p in outputs if not Path(p).exists()]
if missing:
    print("⚠️  Missing:")
    for p in missing:
        print(f"   - {p}")

out_zip = Path("/kaggle/working/outputs.zip")
with zipfile.ZipFile(out_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for p in outputs:
        if Path(p).exists():
            z.write(p, arcname=Path(p).name)

print("✅ Outputs ready:")
for p in [str(out_zip)] + outputs:
    print(f"  {'✅' if Path(p).exists() else '❌'} {p}")
